# BatchNorm 与 Dropout

## 1. 背景：神经网络的两个常见问题

训练神经网络时，初学者最容易遇到这两个问题：

| 问题           | 具体表现                                      | 根本原因                             |
| -------------- | --------------------------------------------- | ------------------------------------ |
| **训练不稳定** | loss 震荡、收敛很慢、需要极小的学习率才不爆炸 | 每层接收到的输入分布随训练不断漂移   |
| **过拟合**     | 训练集 acc 95%，验证集 acc 70%，差距越来越大  | 网络"记住了训练数据"而非学到通用规律 |

**BatchNorm** 针对第一个问题，**Dropout** 针对第二个问题。

## 2. BatchNorm 批归一化

### 2.1 问题根源：Internal Covariate Shift

神经网络是一层接一层的。训练时，前面几层的权重一直在更新，导致**后面每层接收到的输入分布一直在变**。

打个比方：你在教一个学生做题，但每节课给的题目格式完全不同——有时候全是分数，有时候全是负数，有时候数字特别大。这个学生要同时适应"格式"和"解题方法"，学习效率自然很低。

这就是 **Internal Covariate Shift（内部协变量偏移）**。

### 2.2 BN 的解决思路

在每层计算前，先把输入**归一化**成均值为 0、方差为 1 的分布，让每层只需要专注学"解题方法"，不用适应"输入格式"的变化。

### 2.3 BN 的计算过程（一个 batch 内）

```
输入：一个 batch 的数据，shape = (batch_size, features)

Step 1：计算这个 batch 的均值和方差
        μ = mean(x)          # 对 batch 维度求均值
        σ² = var(x)          # 对 batch 维度求方差

Step 2：归一化
        x̂ = (x - μ) / √(σ² + ε)
        # ε 是极小值（如 1e-5），防止分母为零

Step 3：缩放和平移（可学习）
        y = γ · x̂ + β
        # γ（gamma）和 β（beta）是网络自动学习的参数
```

**为什么要有 Step 3？**
如果死板地把所有层都压成均值0方差1，反而限制了网络的表达能力。
γ 和 β 让网络**自己决定**归一化后要不要还原，以及还原成什么样的分布。

### 2.4 训练 vs 推理：两种模式

这是 BN 最需要理解的一点：

| 阶段       | 使用的均值/方差                  | 原因                                     |
| ---------- | -------------------------------- | ---------------------------------------- |
| **训练时** | 当前 batch 的统计量（动态）      | 有足够多的样本可以算                     |
| **推理时** | 训练过程中的**滑动平均**（固定） | 推理可能只有 1 张图，无法算 batch 统计量 |

所以推理前必须调用 `model.eval()`，否则 BN 会拿当前单张图片算统计量，结果完全错误。

### 2.5 BN 的实际效果

- **收敛更快**：可以使用更大的学习率（大 3~5 倍也没问题）
- **对初始化不敏感**：权重初始化好坏影响变小
- **轻微正则化**：batch 统计量本身有随机性，起到类似 Dropout 的效果
- **梯度更稳定**：减少梯度消失/爆炸的风险

### 2.6 代码：怎么加

In [ ]:
import torch.nn as nn

# ---- MLP 中使用 BatchNorm1d ----
# 输入是一维特征向量，如 (batch, 256)
model=nn.Sequential(
    nn.Linear(784,256),
    nn.BatchNorm1d(256),     # 参数 = 上一层输出的特征数
    nn.ReLU(),
    nn.Linear(256,128),
    nn.BatchNorm1d(128),
    nn.ReLU(),
    nn.Linear(128,10)    # 输出层不加 BN
)

# ---- CNN 中使用 BatchNorm2d ----
# 输入是特征图，如 (batch, channels, H, W)
nn.Conv2d(32,64,kernel_size=3,padding=1)
nn.BatchNorm2d(64)         # 参数 = channel 数
nn.ReLU()



**放置顺序（推荐）**：`Linear/Conv → BN → 激活函数`

> 注：也有人把 BN 放在激活函数之后，两种方式都有人用，但放在激活前是更主流的做法。

BatchNorm1d 处理的是向量或一维序列（沿 batch 及可能有的长度轴归一化）；
BatchNorm2d 处理的是二维特征图（沿 batch、高、宽三个轴归一化），保留通道轴。

## 3. Dropout

### 3.1 问题根源：过拟合

过拟合的本质是：网络把训练数据"背下来了"，而不是学到了通用的规律。

一个典型症状：某几个神经元承担了大量工作，整个网络严重依赖它们。训练集上这几个神经元"完美适配"了，但换一组数据就不行了。

### 3.2 Dropout 的解决思路

训练时，**随机把一部分神经元的输出置为 0**（"关掉"它们），每个 step 关掉的神经元都不同。

打个比方：一个 10 人的团队备考，但每次训练随机让 5 人请假。结果：

- 每个人都不能依赖队友，必须独立掌握知识
- 整个团队学到了更分散、更冗余的知识表示

考试时（推理），10 人全部上阵，整体实力反而更强。

### 3.3 Dropout 的计算过程

```
训练时：
  - 以概率 p 把神经元输出置为 0（"丢弃"）
  - 剩余输出除以 (1-p)，做数值缩放

推理时：
  - 所有神经元正常工作
  - 不做任何丢弃操作
```

**为什么训练时要除以 (1-p)？**

假设 p=0.5，训练时平均只有 50% 的神经元激活，输出期望值是正常的一半。
推理时 100% 激活，如果不缩放，输出数值会突然变成训练时的 2 倍，导致结果混乱。
除以 (1-p) = 0.5，让训练和推理的期望输出保持一致。

PyTorch 自动处理这个缩放，你不需要手动计算。

### 3.4 p 值怎么选

`p` 是丢弃概率，p=0.5 表示每个神经元有 50% 概率被关掉。

| 位置             | 推荐 p 值           | 说明                       |
| ---------------- | ------------------- | -------------------------- |
| 全连接隐藏层     | **0.5**             | 最常用的默认值             |
| 靠近输出的隐藏层 | 0.2 ~ 0.3           | 越靠近输出越轻柔           |
| 卷积层           | 0.1 ~ 0.2（或不加） | 卷积层参数少，过拟合风险低 |
| 输出层           | **不加**            | 输出层不能丢弃             |

> p 越大 = 正则化越强 = 越不容易过拟合，但也越容易欠拟合（训练集都学不好）。
> 如果模型训练 acc 也很低，考虑减小 p。

### 3.5 代码：怎么加

In [ ]:
model=nn.Sequential(
    nn.Linear(784,256),
    nn.ReLU(),
    nn.Dropout(p=0.5),      # 放在激活函数之后

    nn.Linear(256,128),
    nn.ReLU(),
    nn.Dropout(p=0.5),

    nn.Linear(128,10)     # 输出层不加 Dropout
)

**放置顺序**：`Linear → 激活函数 → Dropout`

## 4. 两者对比与配合使用

### 4.1 核心对比

| 维度           | BatchNorm                    | Dropout                      |
| -------------- | ---------------------------- | ---------------------------- |
| **解决问题**   | 训练不稳定、收敛慢           | 过拟合                       |
| **作用阶段**   | 训练和推理都有效（行为不同） | 只在训练时有效（推理时关闭） |
| **推荐位置**   | 激活函数之前                 | 激活函数之后                 |
| **可学习参数** | 有（γ 和 β）                 | 无                           |
| **关键超参数** | 基本不用调                   | p 值需要根据过拟合程度调整   |

### 4.2 同时使用时的顺序

```
Linear → BN → ReLU → Dropout
```

完整示例：

```python
nn.Linear(256, 128),
nn.BatchNorm1d(128),
nn.ReLU(),
nn.Dropout(p=0.5),
```

### 4.3 同时使用的注意事项

BN 和 Dropout 同时使用有时会互相干扰（这是一个已知的研究问题）：

- BN 依赖 batch 的统计量，Dropout 改变了激活值的分布，可能影响 BN 的统计准确性
- 实践中：**如果加了 BN，Dropout 的效果可能变弱**，可以考虑减小 p 或只用 BN

对于你当前的实验，三组分别测试即可，不需要同时使用两者。

## 5. 实验代码：三组模型定义

In [ ]:
import torch
import torch.nn as nn

# ============================
# 组 1：Baseline（不加任何正则化）
# ============================
class Baseline(nn.Module):
    def __init__(self, input_dim=784, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# ============================
# 组 2：+BN（加 BatchNorm）
# ============================
class WithBN(nn.Module):
    def __init__(self, input_dim=784, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),        # Linear 之后，激活之前
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)


# ============================
# 组 3：+Dropout（加 Dropout）
# ============================
class WithDropout(nn.Module):
    def __init__(self, input_dim=784, num_classes=10, p=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(p=p),            # 激活之后
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)